# Decoder-only models: GPT family

A decoder-only LM learns one repeated skill: predict the next token using only the prefix.

Decoder-only models keep Transformer blocks but use a causal mask. The same prefix product supports scoring, autocomplete, sampling, and chat-style continuation. Save a copy to Drive to edit.

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

RNG = np.random.default_rng(92)


def softmax(logits):
    values = np.asarray(logits, dtype=float)
    shifted = values - values.max(axis=-1, keepdims=True)
    exp_values = np.exp(shifted)
    return exp_values / exp_values.sum(axis=-1, keepdims=True)


def causal_mask(length):
    return np.tril(np.ones((length, length), dtype=bool))


def decoder_ladder():
    return [
        {"name": "D1 one prefix", "prefix": ["write", "a", "strong", "ad"], "targets": ["now"], "probs": [0.8, 0.5, 0.25]},
        {"name": "D2 clean next-token mini-corpus", "prefix": ["fresh", "creative", "wins"], "targets": ["clicks", "today"], "probs": [0.7, 0.6, 0.5]},
        {"name": "D3 noisy ambiguous prefixes", "prefix": ["brand", "brand", "launch", "offer", "launch"], "targets": ["video", "now"], "probs": [0.55, 0.45, 0.35]},
        {"name": "D4 tiny text snippets", "prefix": ["customer", "asks", "for", "budget", "help", "because"], "targets": ["cost", "rose"], "probs": [0.6, 0.52, 0.41]},
        {"name": "D5 longer prompts with length bias", "prefix": ["summarize", "campaign", "results", "for", "a", "long", "launch", "with", "mixed", "creative"], "targets": ["carefully", "and", "briefly"], "probs": [0.55, 0.5, 0.48, 0.42]},
    ]


TOKEN_SCORES = {
    "now": 2.0,
    "clicks": 1.8,
    "today": 1.4,
    "video": 1.2,
    "cost": 1.1,
    "carefully": 0.9,
    "briefly": 0.8,
}

## The concept, built once: causal next-token scoring

Decoder-only models factor text left-to-right:
$$p(x_{1:T})=\prod_{t=1}^{T}p(x_t\mid x_{\lt t}).$$
A causal mask at $T=4$ permits only $1+2+3+4=10$ visible query-key pairs, not all 16.

In [ ]:
def causal_next_token_lm(prefix, candidate_tokens, temperature=1.0):
    length = len(prefix)
    mask = causal_mask(length)
    visible_pairs = int(mask.sum())
    logits = np.array([TOKEN_SCORES.get(token, 0.2) for token in candidate_tokens], dtype=float)
    probabilities = softmax(logits / temperature)
    prediction = candidate_tokens[int(np.argmax(probabilities))]
    return {
        "mask": mask,
        "visible_pairs": visible_pairs,
        "probabilities": probabilities,
        "prediction": prediction,
    }

The lesson numbers test the mechanism: sequence probability $0.8\times0.5\times0.25=0.100$, temperature $0.5$ turns logits $[2,1,0]$ into top probability $0.867$, and generating 6 tokens means 6 model calls.

In [ ]:
built = causal_next_token_lm(["a", "b", "c", "d"], ["now", "clicks", "other"])
sequence_probability = 0.8 * 0.5 * 0.25
temp_probs = softmax(np.array([2.0, 1.0, 0.0]) / 0.5)
generation_calls = 6

assert built["visible_pairs"] == 10
assert round(sequence_probability, 3) == 0.100
assert round(float(temp_probs[0]), 3) == 0.867
assert generation_calls == 6

print("causal visible pairs", built["visible_pairs"])
print("sequence probability", round(sequence_probability, 3))
print("temperature top probability", round(float(temp_probs[0]), 3))
print("generation calls", generation_calls)

## The dataset ladder

D1-D5 increases prompt length, ambiguity, real-ish text, and length-bias pressure while using the same causal scorer.

In [ ]:
ladder = decoder_ladder()
for rung in ladder:
    print(rung["name"], "context", len(rung["prefix"]), "targets", len(rung["targets"]), "sample", rung["prefix"][:5])

In [ ]:
rows = []
candidates = ["now", "clicks", "today", "video", "cost", "carefully", "briefly"]
for rung in ladder:
    result = causal_next_token_lm(rung["prefix"], candidates)
    nll = -sum(math.log(max(p, 1e-12)) for p in rung["probs"]) / len(rung["probs"])
    perplexity = math.exp(nll)
    rows.append({
        "name": rung["name"],
        "context": len(rung["prefix"]),
        "perplexity": perplexity,
        "mask": result["mask"],
        "prediction": result["prediction"],
    })
for row in rows:
    print(f"{row['name']}: context={row['context']} perplexity={row['perplexity']:.3f} prediction={row['prediction']}")

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
for idx, row in enumerate(rows):
    axes[0, idx].imshow(row["mask"], cmap="Greys", vmin=0, vmax=1)
    axes[0, idx].set_title(f"D{idx + 1} mask")
    axes[1, idx].bar(["ppl"], [row["perplexity"]], color="darkorange")
    axes[1, idx].set_title(row["prediction"])
fig.suptitle("Causal-mask panels and per-rung perplexity")
plt.figure(figsize=(6, 3))
plt.plot([row["context"] for row in rows], [row["perplexity"] for row in rows], marker="o")
plt.xlabel("context length")
plt.ylabel("perplexity")
plt.grid(True)

## Pitfall on D5: leaking future tokens and length bias

Without the causal mask, a scorer can peek at answers and report an invalid low loss. Full-sentence probabilities also punish longer outputs, so compare average log probability per token.

In [ ]:
hard = ladder[-1]
leaked_probabilities = [min(0.99, p + 0.25) for p in hard["probs"]]
leaked_loss = -sum(math.log(p) for p in leaked_probabilities) / len(leaked_probabilities)
causal_loss = -sum(math.log(p) for p in hard["probs"]) / len(hard["probs"])
short_probability = 0.55 * 0.52
long_probability = 0.62 * 0.60 * 0.58 * 0.56
short_avg = math.exp((math.log(0.55) + math.log(0.52)) / 2)
long_avg = math.exp((math.log(0.62) + math.log(0.60) + math.log(0.58) + math.log(0.56)) / 4)
print("invalid leaked loss", round(leaked_loss, 3))
print("valid causal loss", round(causal_loss, 3))
print("raw probabilities short vs long", round(short_probability, 3), round(long_probability, 3))
print("length-normalized scores short vs long", round(short_avg, 3), round(long_avg, 3))

## Evaluate it + practice

            - Metric: perplexity or normalized negative log-likelihood; compare to the no-skill baseline, unigram next-token frequencies.
            - Cheap sanity check: causal mask has triangular sum T(T+1)/2.
            - Ablation: replace the causal mask with all-ones and see invalid loss drop.
            - Failure signals: future leakage, overconfident low temperature, or raw probability favoring short text.
            - Practice: Change temperature and plot the top-token probability.
- Practice: Add a longer D5 candidate and compare raw vs normalized score.
- Practice: Inspect how many model calls are needed for 12 generated tokens.